# box_target_strong: обучение

Настройки и вывод обучения взяты из `train_box_target_strong.ipynb`. Сохранённый вывод: `training.txt`; данные доступных итераций: `training_metrics.csv`. Colab обрезал текст до последних 5000 строк; ранние итерации в этот CSV не входят.
Это компактная копия: убраны загрузочные виджеты, встроенные медиа и старые ячейки инференса, ошибочно проверявшие baseline. Вывод обучения сокращён до последней итерации; значения не изменялись.
Установка зависимостей объединена в одну ячейку с устранением конфликта плагинов JAX. После установки перезапустите сеанс и продолжайте со следующей ячейки; установка использует текущие версии, точный pip freeze исходной сессии не предоставлен.

Этот ноутбук запускает ОБУЧЕНИЕ ЗАНОВО. Для проверки готовой модели используйте `../../evaluate_colab.ipynb`.

In [ ]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cu124
%pip install -q "playground[learning]" "imageio[ffmpeg]" tensorboard
%pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt jax-cuda13-plugin jax-cuda13-pjrt
%pip install -U "jax[cuda12]"

После установки перезапустите сеанс Colab. Затем выполняйте ячейки ниже.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
LOG_ROOT = "/content/drive/MyDrive/physicalai_sprint3"
os.makedirs(LOG_ROOT, exist_ok=True)
os.chdir(LOG_ROOT)
os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

In [ ]:
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""

import json
import os
import time
from datetime import datetime
from types import SimpleNamespace

import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner

jax.config.update("jax_default_matmul_precision", "highest")

# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 2048
MAX_ITERS = 1001
SAVE_INTERVAL = 50
SEED = 1
EXP_NAME = "box_target_strong"
RESUME = None
DEVICE = "cuda:0"

# --- config.py ---
ENV_NAME = "PandaPickCube"

ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0


def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg


def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("logs", name)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir


args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)

device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0

log_dir = resolve_log_dir(args.exp_name, args.resume)

print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()

env_cfg = registry.get_default_config(ENV_NAME)
env_cfg.reward_config.scales.box_target = 40.0

print("reward_config.scales:", env_cfg.reward_config.scales)
raw_env = registry.load(ENV_NAME, config=env_cfg)

brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}

cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval

config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")

print("learning_rate:", cfg_dict["algorithm"]["learning_rate"])
experiment = {
    "exp_name": EXP_NAME,
    "num_envs": NUM_ENVS,
    "max_iters": MAX_ITERS,
    "seed": SEED,
    "resume": RESUME,
    "learning_rate": cfg_dict["algorithm"]["learning_rate"],
    "reward_scales": dict(env_cfg.reward_config.scales),
}
with open(os.path.join(log_dir, "experiment.json"), "w", encoding="utf-8") as f:
    json.dump(experiment, f, ensure_ascii=False, indent=2, default=str)

runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)

if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)

print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()

t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start

print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)

Learning iteration 1000/1001                           

                            Total steps: 82001920 
                       Steps per second: 68629 
                        Collection time: 0.836s 
                          Learning time: 0.358s 
                        Mean value loss: 1110.9101
                    Mean surrogate loss: -0.0056
                      Mean entropy loss: -23.8486
                            Mean reward: 4453.04
                    Mean episode length: 150.00
                          Learning rate: 0.0002601
                        Mean policy std: 4.94
                Mean episode box_target: 0.7797
               Mean episode gripper_box: 0.9068
        Mean episode no_floor_collision: 0.9917
             Mean episode out_of_bounds: 0.0000
         Mean episode robot_target_qpos: 0.1426
--------------------------------------------------------------------------------
                         Iteration time: 1.19s
                           Time el